# DAF-12: Weather History — Two Versions of the Past

### 🎬 The story

Asha wants tomorrow’s Delhi PM2.5 forecast to use weather information. At 18:00, she can use a weather forecast for tomorrow—but she cannot know tomorrow’s actual weather yet. If we train using weather that actually happened, the model gets information it would not have in production.

DAF-12 collects both versions so DAF-13 can decide which one is appropriate. We will save the API responses unchanged, check their hourly coverage, and compare one week of wind speed. We will **not** join weather to the PM2.5 table in this notebook.

### 🗺️ The journey

```text
1. Resolve R K Puram’s station coordinates from stations.csv
2. Request archive weather and historical forecast weather
3. Save each raw response as JSON; skip existing files on rerun
4. Check hourly coverage across the requested date range
5. Compare one week of wind speed from both sources
6. Record observations for DAF-13’s source decision
```

We’ll build and verify one step at a time. Each step will have an explanation first, your code next, and an output-reading note after it.

# 🟨 Explanation_2 — Step 1: Find the right station coordinates

### 🎬 How we got here

We know the weather API needs latitude and longitude. The ticket says those coordinates must come from `data/raw/stations.csv`, not be typed directly into the collector. Before calling an API, we need to establish which station record represents the R K Puram sensor used by this project.

### 😖 The problem

A station ID may not identify just one row. In this project, `location_id=17` appears twice: one record is for the sensor used in the current PM2.5 data, and another is an older sensor. Picking the first matching row without checking could attach the wrong sensor history to this collection.

```text
stations.csv
    │
    ├── location_id 17, sensor_id 12234787
    │      R K Puram; readings span 2025-02-19 to 2026-09-11
    │
    └── location_id 17, sensor_id 35
           R K Puram; older readings span 2016-02-05 to 2018-02-22
```

Both rows currently show the same coordinates, but their sensor histories differ. We should identify the intended record by matching the project’s station and date range—not by assuming the first row is correct.

### 💡 The idea

Treat `stations.csv` like a service’s configuration registry: look up the entity, inspect possible matches, then select the record that matches the workload. Here, the workload is the R K Puram PM2.5 period from **2025-02-19 through 2026-09-11**.

The selected record should provide the coordinates used by both weather API requests. We also keep the sensor ID and coverage visible so the choice is explainable later.

The analogy is not exact: a CSV is not a live service registry, and these coordinates identify a place rather than a running service. The useful lesson is to inspect ambiguous configuration instead of silently taking the first match.

### 🔍 What this actually is

This is a **metadata lookup and validation** step. It does not download weather. It verifies the station identity and extracts the latitude and longitude needed by the collector.

The matching record in the current file is: `location_id=17`, `sensor_id=12234787`, latitude `28.563262`, longitude `77.186937`. Its recorded coverage matches the date range in DAF-12. The older `sensor_id=35` row has a different historical period.

### What your code cell should do

Write a small inspection cell that reads `data/raw/stations.csv`, displays all rows for location 17, and makes the selected current sensor record and its coordinates clear. Keep paths relative to the project root. Do not call either weather API yet.

Expected inspection result:

```text
Selected location: R K Puram, Delhi - DPCC
Selected sensor:   12234787
Coordinates:       28.563262, 77.186937
Coverage:          2025-02-19 ... 2026-09-11
```

### What breaks if we skip this?

We could request weather for the wrong coordinates or fail to explain which station record the data represents. The coordinates happen to match for these two rows today, but relying on that coincidence would make the collector fragile if the metadata changes.

### 🤔 Before you write the code

Why should we display both matching station rows before selecting one, even though their coordinates currently match? Write your guess, then build the inspection cell and compare it with the output.

### ✅ Checkpoint

1. Which record matches the PM2.5 period used by this project?
2. What two values will the weather APIs need from that record?
3. Why should the collector not hard-code those values?

**Answers:** The current record is sensor `12234787`, whose coverage matches the project period. The API needs latitude and longitude. Reading them from the station metadata keeps the collector tied to the project’s configured station rather than embedding assumptions in code.

### Final takeaway

```text
Find station metadata → resolve duplicate records → verify dates → use its coordinates
```

This step gives both API requests a traceable location. Next, we’ll understand what the archive and historical-forecast endpoints each mean before making a request.

In [1]:
from pathlib import Path

import pandas as pd

project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "18_Projects/01_Delhi_Air_Forecast/data/raw/stations.csv").is_file()
        or (candidate / "data/raw/stations.csv").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find data/raw/stations.csv from this notebook's working directory.")

stations_path = project_root / "data/raw/stations.csv"
if not stations_path.is_file():
    stations_path = project_root / "18_Projects/01_Delhi_Air_Forecast/data/raw/stations.csv"

# stations.csv is small, but usecols keeps this lookup focused on metadata needed here.
stations = pd.read_csv(
    stations_path,
    usecols=[
        "location_id",
        "sensor_id",
        "name",
        "latitude",
        "longitude",
        "first_reading",
        "last_reading",
    ],
)
stations = stations.loc[stations["location_id"].eq(17)]

requested_start = "2025-02-19"
requested_end = "2026-09-11"
station_rows = stations.copy()
station_rows["coverage_start"] = station_rows["first_reading"].astype("string").str[:10]
station_rows["coverage_end"] = station_rows["last_reading"].astype("string").str[:10]

display(
    station_rows[
        ["location_id", "sensor_id", "name", "latitude", "longitude", "coverage_start", "coverage_end"]
    ]
)

matching_rows = station_rows.loc[
    station_rows["coverage_start"].le(requested_start)
    & station_rows["coverage_end"].ge(requested_end)
]
if len(matching_rows) != 1:
    raise ValueError(
        f"Expected one location-17 record covering {requested_start} through {requested_end}; "
        f"found {len(matching_rows)}. Inspect the station metadata before continuing."
    )

selected_station = matching_rows.iloc[0]
latitude = float(selected_station["latitude"])
longitude = float(selected_station["longitude"])

print(f"Selected location: {selected_station['name']}")
print(f"Selected sensor:   {selected_station['sensor_id']}")
print(f"Coordinates:       {latitude}, {longitude}")
print(f"Coverage:          {selected_station['coverage_start']} ... {selected_station['coverage_end']}")

,location_id,sensor_id,name,latitude,longitude,coverage_start,coverage_end
3,17,12234787,"R K Puram, Delhi - DPCC",28.563262,77.186937,2025-02-19,2026-09-11
4,17,35,"R K Puram, Delhi - DPCC",28.563262,77.186937,2016-02-05,2018-02-22


Selected location: R K Puram, Delhi - DPCC
Selected sensor:   12234787
Coordinates:       28.563262, 77.186937
Coverage:          2025-02-19 ... 2026-09-11


### 👀 Read the result

The table shows two records for location 17. Their coordinates happen to match, but their coverage periods do not: sensor `12234787` covers **2025-02-19 through 2026-09-11**, while sensor `35` is from 2016–2018.

The code selected exactly one record whose coverage contains the requested dates, then exposed its latitude and longitude as `28.563262, 77.186937`. This confirms the location inputs for the weather requests without contacting either API yet.

# 🟨 Explanation_2 — Step 2: Request both versions of weather history

### 🎬 How we got here

Step 1 resolved the current R K Puram station and gave us traceable coordinates. Now we can ask Open-Meteo for weather at that exact location, over the same period as the PM2.5 data.

### 😖 The problem

There are two useful histories, and they answer different questions:

- **Archive weather** tells us what actually happened at R K Puram.
- **Historical forecast weather** tells us what a forecast would have predicted at the time.

The archive version is usually cleaner because it describes the realised weather. But at 18:00, when Asha predicts tomorrow's PM2.5, tomorrow's realised weather does not exist yet. If we train or test with archive weather, the model may appear better because it is seeing information that production will not have.

### 💡 The idea

Ask both services for the same coordinates, date range, timezone, and hourly variables. Keeping the requests identical means that any later difference is about the source, not about different parameters. We will preserve each response as raw JSON; DAF-13 will decide which version belongs in the modelling table.

The analogy is a staging replay: archive weather is like inspecting the final production outcome, while historical forecast weather is like inspecting the prediction that was available before the outcome. The analogy breaks down because weather forecasts are numerical simulations, not software deployments, but the information-timing problem is the same.

### 🔍 What this actually is

This is an **API contract check**, not feature engineering. We are checking that both endpoints accept the same request shape and return hourly records for the selected station. We are deliberately not joining weather to PM2.5, calculating daily features, or choosing a source yet.

The request asks for temperature, humidity, wind speed, wind direction, precipitation, and surface pressure in `Asia/Kolkata`. The timezone matters because the PM2.5 decision point is 18:00 local time.

### What your code cell should do

1. Build one shared parameter dictionary from `latitude`, `longitude`, and the DAF-12 date range.
2. Send that request to both Open-Meteo endpoints.
3. Keep each decoded response in a separate raw object.
4. Check the HTTP status, returned timezone, hourly keys, and number of hourly timestamps.

### What breaks if we skip this?

We could compare mismatched periods, accidentally use UTC while reasoning in India Standard Time, or discover only much later that one endpoint does not return the variables needed by the model. Most importantly, we would lose the evidence needed for DAF-13's train/serve decision.

### 🤔 Before you run the code

Which response do you expect to be easier to use for explaining what really happened: the archive response or the historical-forecast response? Make a prediction first, then inspect the returned metadata.

### ✅ Checkpoint

1. In one sentence, what does the archive API provide?
2. In one sentence, what does the historical forecast API provide?
3. Why can archive weather make a test score look unrealistically good?

**Answers:** The archive API provides weather that actually happened. The historical forecast API provides weather that was forecast at the time. Archive weather can make the score look too good because it gives the model realised future conditions that would not be available when tomorrow's forecast is made.

### Final takeaway

```text
One location + one date range + one variable list → two weather information timelines
```

Next, Step 3 will save each response unchanged under `data/raw/openmeteo/{source}/` and make the collection idempotent.

# 👀 Visual explanation — how the threaded download works

### The original problem

One request for 19 months is a large package. It may be slow or may be refused. We split the same work into smaller date ranges and run a few requests at the same time.

```text
Full period: 2025-02-19 ───────────────────────────────────────── 2026-09-11

Instead of one large request:

[ archive + forecast: 19 months ]

We create seven 90-day chunks:

[Feb-May] [May-Aug] [Aug-Nov] [Nov-Feb] [Feb-May] [May-Aug] [Aug-Sep]
    │         │         │         │         │         │         │
    └─────────┴─────────┴─────────┴─────────┴─────────┴─────────┴── archive
    └─────────┴─────────┴─────────┴─────────┴─────────┴─────────┴── forecast

Total: 14 independent API requests
```

### What “four threads” means

A **thread** is one worker that can wait for an API response while other workers do useful waiting too. `max_workers=4` means that at most four requests are in flight at one time:

```text
Thread 1:  archive Feb-May  ──────── response ──┐
Thread 2:  archive May-Aug  ───────── response ─┤
Thread 3:  forecast Feb-May ─────── response ────┤── store results
Thread 4:  forecast May-Aug ──────── response ──┘

When one finishes, that worker takes the next waiting chunk.
```

This is useful because the program spends most of its time waiting for the network. It is not four Python programs changing the same data at once. Each worker only downloads one independent chunk, and the main notebook cell stores the completed result.

### The code flow

```text
make_date_chunks()
        │
        ▼
14 jobs: (source, endpoint, start_date, end_date)
        │
        ▼
ThreadPoolExecutor(max_workers=4)
        │
        ├── submit job 1 ──► Future 1 ──► waiting for API
        ├── submit job 2 ──► Future 2 ──► waiting for API
        ├── submit job 3 ──► Future 3 ──► waiting for API
        └── submit job 4 ──► Future 4 ──► waiting for API
                         ...
        │
        ▼
as_completed(futures) gives whichever job finishes next
        │
        ▼
raw_weather_chunks[source][chunk_start] = payload
```

### The important Python pieces

- `ThreadPoolExecutor(max_workers=4)` creates four reusable workers.
- `executor.submit(...)` starts a job and immediately gives back a **Future**, which is a promise that a result will arrive later.
- `as_completed(futures)` lets us handle results in completion order. The fastest request does not wait for the slowest request.
- `future.result()` retrieves the result. If an API request failed, this line raises the real error instead of hiding it.
- `raw_weather_chunks` keeps archive and forecast responses separate. The payloads are not merged or modified.

### Why not use unlimited threads?

Unlimited requests could overload the API, trigger rate limits, or make failures harder to diagnose. Four workers are a controlled compromise: faster than one request at a time, but still polite and predictable.

The result is not one final dataframe yet. It is fourteen raw JSON payloads, grouped by source and date chunk. Step 3 will save those payloads unchanged to disk.

In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date

import requests

weather_start = "2025-02-19"
weather_end = "2026-09-11"
hourly_variables = ",".join(
    [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "wind_direction_10m",
        "precipitation",
        "surface_pressure",
    ]
)

shared_parameters = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": hourly_variables,
    "timezone": "Asia/Kolkata",
}

endpoints = {
    "archive": "https://archive-api.open-meteo.com/v1/archive",
    "forecast": "https://historical-forecast-api.open-meteo.com/v1/forecast",
}


def make_date_chunks(start_date, end_date, chunk_days=90):
    chunks = []
    current = pd.Timestamp(start_date)
    final_date = pd.Timestamp(end_date)
    while current <= final_date:
        chunk_end = min(current + pd.Timedelta(days=chunk_days - 1), final_date)
        chunks.append((current.date().isoformat(), chunk_end.date().isoformat()))
        current = chunk_end + pd.Timedelta(days=1)
    return chunks


def fetch_weather_chunk(source, endpoint, chunk_start, chunk_end):
    parameters = {
        **shared_parameters,
        "start_date": chunk_start,
        "end_date": chunk_end,
    }
    response = requests.get(endpoint, params=parameters, timeout=(10, 60))
    response.raise_for_status()
    payload = response.json()

    hourly = payload["hourly"]
    expected_hours = len(
        pd.date_range(
            chunk_start,
            pd.Timestamp(chunk_end) + pd.Timedelta(days=1) - pd.Timedelta(hours=1),
            freq="h",
        )
    )
    assert payload["timezone"] == "Asia/Kolkata"
    assert set(hourly_variables.split(",")).issubset(hourly)
    assert len(hourly["time"]) == expected_hours, (
        f"{source} {chunk_start} to {chunk_end}: "
        f"returned {len(hourly['time'])}; expected {expected_hours}"
    )
    return source, chunk_start, chunk_end, payload


chunks = make_date_chunks(weather_start, weather_end)
raw_weather_chunks = {source: {} for source in endpoints}
requests_to_send = [
    (source, endpoint, chunk_start, chunk_end)
    for source, endpoint in endpoints.items()
    for chunk_start, chunk_end in chunks
]

# Four workers keep the download bounded and reduce the chance of API throttling.
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(fetch_weather_chunk, *request) for request in requests_to_send]
    for future in as_completed(futures):
        source, chunk_start, chunk_end, payload = future.result()
        raw_weather_chunks[source][chunk_start] = payload
        print(f"Downloaded {source}: {chunk_start} through {chunk_end}")

for source in raw_weather_chunks:
    raw_weather_chunks[source] = dict(sorted(raw_weather_chunks[source].items()))
    print(f"{source.title()} API: {len(raw_weather_chunks[source])} raw chunks")

print(
    f"Downloaded the complete period {weather_start} through {weather_end} "
    f"in {len(requests_to_send)} bounded parallel requests."
)

Downloaded archive: 2025-02-19 through 2025-05-19
Downloaded archive: 2025-08-18 through 2025-11-15
Downloaded archive: 2025-11-16 through 2026-02-13
Downloaded archive: 2025-05-20 through 2025-08-17
Downloaded archive: 2026-08-13 through 2026-09-11
Downloaded archive: 2026-02-14 through 2026-05-14
Downloaded forecast: 2025-02-19 through 2025-05-19
Downloaded archive: 2026-05-15 through 2026-08-12
Downloaded forecast: 2025-05-20 through 2025-08-17
Downloaded forecast: 2025-08-18 through 2025-11-15
Downloaded forecast: 2025-11-16 through 2026-02-13
Downloaded forecast: 2026-02-14 through 2026-05-14
Downloaded forecast: 2026-08-13 through 2026-09-11
Downloaded forecast: 2026-05-15 through 2026-08-12
Archive API: 7 raw chunks
Forecast API: 7 raw chunks
Downloaded the complete period 2025-02-19 through 2026-09-11 in 14 bounded parallel requests.


# 🟨 Explanation_2 — Step 3: What we are collecting and why

## 🎬 The business question

Asha wants to predict tomorrow's PM2.5 at R K Puram. Weather matters because still air can let pollution build up, while wind can disperse it.

But there is a timing problem:

```text
At 18:00 today, predicting tomorrow

                         tomorrow has not happened yet
                                      │
                                      ▼
              We can use a weather forecast, not tomorrow's actual weather
```

If we train a model with tomorrow's **actual** weather but serve it with a **forecast**, the training experiment gets better information than production. That is a form of information leakage.

## Why call two APIs?

The two APIs describe two different versions of the same weather timeline:

```text
                         Same place and same dates
                         R K Puram, Delhi
                                  │
                ┌─────────────────┴─────────────────┐
                │                                   │
                ▼                                   ▼
      ARCHIVE WEATHER API                 HISTORICAL FORECAST API
      What actually happened              What was forecast at the time
      ----------------------              ------------------------------
      "The wind really was                "The forecast available then
       4.2 km/h at 08:00"                  predicted 3.8 km/h"
                │                                   │
                └─────────────────┬─────────────────┘
                                  ▼
                    Compare them later in DAF-13
```

The archive API is useful for understanding the real physical conditions. The historical-forecast API is useful for simulating what information would have been available when a real prediction was made.

We are not choosing the winner yet. We are collecting both so DAF-13 can make that decision with evidence.

## What one API response looks like

Each response contains metadata plus an `hourly` section. Conceptually, the JSON looks like this:

```text
{
  "latitude": 28.576448,
  "longitude": 77.186780,
  "timezone": "Asia/Kolkata",
  "hourly": {
    "time":                 ["2025-02-19T00:00", "2025-02-19T01:00", ...],
    "temperature_2m":       [18.4, 17.9, ...],
    "relative_humidity_2m": [72, 74, ...],
    "wind_speed_10m":       [3.1, 2.8, ...],
    "wind_direction_10m":   [250, 248, ...],
    "precipitation":        [0.0, 0.0, ...],
    "surface_pressure":     [1012.5, 1012.7, ...]
  }
}
```

The arrays line up by position. The first value in every array belongs to the first time:

| Time in India | Temperature | Humidity | Wind speed | Wind direction | Rain | Pressure |
|---|---:|---:|---:|---:|---:|---:|
| 2025-02-19 00:00 | 18.4 °C | 72% | 3.1 km/h | 250° | 0.0 mm | 1012.5 hPa |
| 2025-02-19 01:00 | 17.9 °C | 74% | 2.8 km/h | 248° | 0.0 mm | 1012.7 hPa |
| ... | ... | ... | ... | ... | ... | ... |

For the full period, we expect approximately 13,800 hourly records from each source. The threaded code requests those records in smaller chunks so one huge response does not become a bottleneck.

## What we retrieve

```text
One station
    │
    ├── 2025-02-19 through 2026-09-11
    │       │
    │       └── one record per hour
    │              ├── temperature
    │              ├── humidity
    │              ├── wind speed
    │              ├── wind direction
    │              ├── precipitation
    │              └── surface pressure
    │
    ├── Archive copy: 7 raw JSON chunks
    └── Forecast copy: 7 raw JSON chunks
```

The 14 files are not 14 different datasets. They are two versions of one weather history, split into date ranges for reliable downloading:

```text
archive/
├── location-17-2025-02-19-2025-05-19.json
├── location-17-2025-05-20-2025-08-17.json
└── ... five more chunks

forecast/
├── location-17-2025-02-19-2025-05-19.json
├── location-17-2025-05-20-2025-08-17.json
└── ... five more chunks
```

## What happens now and what happens later

```text
STEP 1                 STEP 2                    STEP 3                 DAF-13
Find station     →     Ask both APIs       →     Save raw JSON      →    Decide source
coordinates            and validate             unchanged                and join carefully
                         responses

                         We are here: raw data collection
```

Step 3 only saves evidence. It does **not**:

- join weather with PM2.5;
- convert hourly weather into daily features;
- select archive or forecast;
- train a model;
- calculate a prediction score.

Those decisions come later. Keeping this step raw and separate lets us inspect exactly what each API returned and reproduce the collection.

## The save operation

```text
In memory                                      On disk
─────────                                      ───────
raw_weather_chunks["archive"]["2025-02-19"] ──► data/raw/openmeteo/archive/...
raw_weather_chunks["forecast"]["2025-02-19"] ─► data/raw/openmeteo/forecast/...

If the target file exists: SKIP
If the target file is missing: WRITE
```

This is called **idempotent** behaviour: running the save cell again produces no duplicate downloads and does not overwrite existing raw evidence.

### 🤔 Before you run the code

Why do we save archive and forecast responses separately instead of combining them immediately? The answer is: we need to preserve the two information timelines until DAF-13 decides which one matches the way the model will operate in production.

## ❓ What question is this statement answering?

> If we train a model with tomorrow's **actual** weather but serve it with a **forecast**, the training experiment gets better information than production. That is a form of information leakage.

The question is:

**Why is using tomorrow's actual weather during training unfair?**

Asha makes the prediction at 18:00 today. At that moment, tomorrow's actual weather is unknown. She can only use a forecast.

```text
Training / testing                 Production at 18:00
------------------                 -------------------
Actual weather tomorrow            Forecast weather tomorrow
        │                                  │
        └──────────────► PM2.5 ◄───────────┘

Training sees the real answer.      Production sees an estimate.
```

If training uses actual future weather, the model receives information that will not exist when the deployed model runs. Its test score can therefore look better than its real production performance.

This is information leakage because future information has entered the model before that information would genuinely be available.

The two weather sources represent the difference:

```text
Archive API              Historical Forecast API
actual weather           forecast available at that time
```

## 🔎 Plain-English explanation: what is information leakage?

Imagine Asha makes a prediction at **18:00 on February 19** for PM2.5 on **February 20**.

At 18:00, she can know:

```text
Available now
├── Today's PM2.5
├── Today's weather
└── A weather forecast for February 20
```

She cannot know the actual February 20 weather yet. That weather will only be known after February 20 finishes.

### The unfair training example

Suppose the training data contains this row:

| Feature used by the model | Value |
|---|---:|
| Actual wind speed on February 20 | 2 km/h |
| Actual PM2.5 on February 20 | High |

The model sees the **actual future weather** and learns that low wind is associated with high PM2.5. The relationship may be real, but the timing is unfair: production will not know the actual wind speed when making the prediction.

```text
TRAINING / TESTING
Actual future weather ─────────► PM2.5
Perfect information               Target

PRODUCTION AT 18:00
Weather forecast ───────────────► PM2.5
Imperfect estimate                Target
```

The training experiment may show an excellent score because it receives better information than the deployed model. After deployment, the score can fall because forecasts contain errors.

### Why this is called leakage

**Information leakage** means information has entered the model that would not be available at the moment the prediction is supposed to be made. It is like allowing someone to see the answer sheet before taking the test: the result looks impressive, but it does not represent the real challenge.

In this project, the two APIs show the difference:

```text
Archive API
└── What actually happened on February 20

Historical Forecast API
└── What a forecast would have predicted before February 20
```

The archive data helps us understand the real weather and investigate relationships. The historical-forecast data helps us simulate production honestly. DAF-13 will decide which source, or combination of sources, is appropriate for the model.

**Important:** this notebook is only collecting the evidence. It is not choosing the final source or training the model yet.

In [7]:
import json

if (project_root / "data/raw/stations.csv").is_file():
    raw_root = project_root / "data/raw/openmeteo"
else:
    raw_root = project_root / "18_Projects/01_Delhi_Air_Forecast/data/raw/openmeteo"

saved_files = []
skipped_files = []

for source, chunks_by_start in raw_weather_chunks.items():
    source_directory = raw_root / source
    source_directory.mkdir(parents=True, exist_ok=True)

    for chunk_start, payload in chunks_by_start.items():
        chunk_end = payload["hourly"]["time"][-1][:10]
        output_path = source_directory / f"location-17-{chunk_start}-{chunk_end}.json"

        if output_path.exists():
            skipped_files.append(output_path)
            continue

        output_path.write_text(
            json.dumps(payload, indent=2) + "\n",
            encoding="utf-8",
        )
        saved_files.append(output_path)

print(f"Saved new files:    {len(saved_files)}")
print(f"Skipped existing:   {len(skipped_files)}")
print(f"Total raw files:    {len(saved_files) + len(skipped_files)}")
print(f"Raw data directory: {raw_root}")

Saved new files:    0
Skipped existing:   14
Total raw files:    14
Raw data directory: /Users/kaliprasad/Documents/MACHINE_LEARNING/18_Projects/01_Delhi_Air_Forecast/data/raw/openmeteo
